# 📊 Exploratory Data Analysis — Datathon Candidate Database---This notebook performs a comprehensive EDA on the synthetic candidate database from the Datathon.  We explore **175 synthetic CV profiles** across 5 roles, examining fit scores, experience, certifications, and edge cases.

In [ ]:
# === Imports & Configuration ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style settings
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print("✅ Libraries loaded successfully.")

In [ ]:
# === Load the Dataset ===
df = pd.read_excel('Datathon Database.xlsx')
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

## 1. Dataset OverviewLet's inspect the schema, data types, and basic statistics.

In [ ]:
# Column names and data types
df.info()

In [ ]:
# Descriptive statistics for numeric columns
df.describe()

In [ ]:
# === Missing Values Analysis ===
# FINDING: Only 2 columns have missing values:
#   - Certification (26 nulls / 14.9%) — some candidates lack formal certs
#   - Surprise / Edge Case (150 nulls / 85.7%) — only 25 "Unusual/Surprise" candidates have this filled

missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df[missing_df['Missing Count'] > 0]

## 2. Categorical Variable Distributions### Key Finding: The dataset is **perfectly balanced** across Role and Field (35 each), and intentionally structured across Candidate Levels.

In [ ]:
# === Role & Field Distribution ===
# FINDING: Exactly 35 candidates per role/field — this is a designed, balanced dataset
# Roles: Project Manager, Financial Analyst, Digital Marketing Specialist, Data Analyst, Backend Developer
# Fields: Management, Finance, Marketing, Tech/Data, Tech/Software

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Role distribution
role_counts = df['Role'].value_counts()
colors_role = sns.color_palette('viridis', len(role_counts))
bars1 = axes[0].barh(role_counts.index, role_counts.values, color=colors_role)
axes[0].set_title('Candidates per Role', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Count')
for bar, val in zip(bars1, role_counts.values):
    axes[0].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, str(val), va='center', fontweight='bold')

# Field distribution
field_counts = df['Field'].value_counts()
colors_field = sns.color_palette('magma', len(field_counts))
bars2 = axes[1].barh(field_counts.index, field_counts.values, color=colors_field)
axes[1].set_title('Candidates per Field', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Count')
for bar, val in zip(bars2, field_counts.values):
    axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, str(val), va='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# === Candidate Level Distribution ===
# FINDING: 50 Strong, 50 Average, 50 Weak, and 25 Unusual/Surprise candidates
# The "Unusual/Surprise" group is intentionally smaller — used to test edge-case reasoning

level_counts = df['Candidate Level'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#9b59b6']
fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(level_counts.values, labels=level_counts.index, 
                                   autopct='%1.1f%%', colors=colors, startangle=90,
                                   textprops={'fontsize': 12}, pctdistance=0.8,
                                   wedgeprops=dict(width=0.5, edgecolor='white', linewidth=2))
for t in autotexts:
    t.set_fontweight('bold')
ax.set_title('Candidate Level Distribution', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# === Expected Recommendation Distribution ===
# FINDING: Maps directly to Candidate Level:
#   Strong → "Recommend" (50), Average → "Maybe / needs further assessment" (50),
#   Weak → "Do not recommend" (50), Unusual → "Conditional / requires follow-up interview" (25)

rec_counts = df['Expected Recommendation'].value_counts()
colors_rec = ['#27ae60', '#f1c40f', '#e74c3c', '#8e44ad']
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(len(rec_counts)), rec_counts.values, color=colors_rec, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(len(rec_counts)))
ax.set_xticklabels(rec_counts.index, rotation=15, ha='right', fontsize=10)
ax.set_title('Expected Recommendation Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
for bar, val in zip(bars, rec_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, str(val), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Numerical Variable Analysis### Key Finding: Years of Experience and Fit Score are **strongly correlated** (r = 0.69)

In [ ]:
# === Years of Experience Distribution ===
# FINDING: Right-skewed; mean = 3.95 years, range 0–12
# Weak candidates cluster at 0–2 years, Strong at 5–9, Unusual/Surprise spans the full range

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['Years Experience'], bins=13, color='#3498db', edgecolor='white', linewidth=1.2, alpha=0.85)
axes[0].axvline(df['Years Experience'].mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f"Mean = {df['Years Experience'].mean():.1f}")
axes[0].axvline(df['Years Experience'].median(), color='#2ecc71', linestyle='--', linewidth=2, label=f"Median = {df['Years Experience'].median():.1f}")
axes[0].set_title('Years of Experience Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Years')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Fit Score Distribution
axes[1].hist(df['Expected Fit Score'], bins=15, color='#e67e22', edgecolor='white', linewidth=1.2, alpha=0.85)
axes[1].axvline(df['Expected Fit Score'].mean(), color='#e74c3c', linestyle='--', linewidth=2, label=f"Mean = {df['Expected Fit Score'].mean():.1f}")
axes[1].axvline(df['Expected Fit Score'].median(), color='#2ecc71', linestyle='--', linewidth=2, label=f"Median = {df['Expected Fit Score'].median():.1f}")
axes[1].set_title('Expected Fit Score Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# === Fit Score by Candidate Level (Box Plot) ===
# FINDING: Clear separation between levels — the scoring system is well-designed:
#   Strong: 82–95 (mean 88.6), Average: 58–76 (mean 66.3),
#   Weak: 25–49 (mean 36.1), Unusual: 45–80 (mean 60.3, high variance)
# The Unusual group overlaps with both Average and Weak — intentionally ambiguous

fig, ax = plt.subplots(figsize=(10, 6))
order = ['Strong', 'Average', 'Unusual/Surprise', 'Weak']
palette = {'Strong': '#2ecc71', 'Average': '#f39c12', 'Unusual/Surprise': '#9b59b6', 'Weak': '#e74c3c'}
sns.boxplot(data=df, x='Candidate Level', y='Expected Fit Score', order=order, palette=palette, ax=ax, linewidth=1.5)
sns.stripplot(data=df, x='Candidate Level', y='Expected Fit Score', order=order, color='black', alpha=0.3, size=4, ax=ax)
ax.set_title('Expected Fit Score by Candidate Level', fontsize=14, fontweight='bold')
ax.set_ylabel('Fit Score')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# === Years Experience by Candidate Level ===
# FINDING: Experience clearly separates Strong (5-9 yrs) from Weak (0-2 yrs)
# Unusual/Surprise has the WIDEST spread (1-12 yrs) — these are the unpredictable candidates

fig, ax = plt.subplots(figsize=(10, 6))
sns.violinplot(data=df, x='Candidate Level', y='Years Experience', order=order, palette=palette, ax=ax, inner='box')
ax.set_title('Years Experience by Candidate Level', fontsize=14, fontweight='bold')
ax.set_ylabel('Years')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# === Scatter: Experience vs Fit Score ===
# FINDING: Strong positive correlation (r = 0.69). However, the "Unusual/Surprise"
# candidates break this pattern — some have many years but modest scores, and vice versa.
# This makes them ideal test cases for AI evaluation robustness.

fig, ax = plt.subplots(figsize=(10, 7))
for level in order:
    subset = df[df['Candidate Level'] == level]
    ax.scatter(subset['Years Experience'], subset['Expected Fit Score'], 
               label=level, color=palette[level], alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

z = np.polyfit(df['Years Experience'], df['Expected Fit Score'], 1)
p = np.poly1d(z)
x_line = np.linspace(0, 12, 100)
ax.plot(x_line, p(x_line), '--', color='gray', linewidth=2, alpha=0.7, label=f'Trend (r={0.69:.2f})')

ax.set_title('Years Experience vs Expected Fit Score', fontsize=14, fontweight='bold')
ax.set_xlabel('Years Experience')
ax.set_ylabel('Expected Fit Score')
ax.legend(title='Candidate Level')
plt.tight_layout()
plt.show()

## 4. Cross-Tabulations & Heatmaps

In [ ]:
# === Fit Score Heatmap: Role × Candidate Level ===
# FINDING: Fit scores are very consistent across roles within the same level.
# No single role has systematically higher or lower scores — the difficulty is uniform.

pivot = df.pivot_table(values='Expected Fit Score', index='Role', columns='Candidate Level', aggfunc='mean')
pivot = pivot[order]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', linewidths=1, linecolor='white',
            ax=ax, vmin=25, vmax=95, cbar_kws={'label': 'Mean Fit Score'})
ax.set_title('Mean Fit Score: Role × Candidate Level', fontsize=14, fontweight='bold')
ax.set_ylabel('')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

In [ ]:
# === Experience Heatmap: Role × Candidate Level ===
# FINDING: Similar pattern — experience levels are consistent across roles.
# Strong candidates average 6-8 years regardless of role. Weak = 0-1 year.

pivot_exp = df.pivot_table(values='Years Experience', index='Role', columns='Candidate Level', aggfunc='mean')
pivot_exp = pivot_exp[order]

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot_exp, annot=True, fmt='.1f', cmap='YlOrBr', linewidths=1, linecolor='white',
            ax=ax, cbar_kws={'label': 'Mean Years Experience'})
ax.set_title('Mean Years Experience: Role × Candidate Level', fontsize=14, fontweight='bold')
ax.set_ylabel('')
ax.set_xlabel('')
plt.tight_layout()
plt.show()

## 5. Education & Certification Analysis

In [ ]:
# === Top Education Backgrounds ===
# FINDING: 21 unique education backgrounds. Most common: BSc Information Systems (13),
# BSc Business Administration (13), BSc Computer Science (12).
# Mix of technical and business degrees reflects the diversity of roles.

edu_counts = df['Education'].value_counts()

fig, ax = plt.subplots(figsize=(12, 7))
colors_edu = sns.color_palette('coolwarm', len(edu_counts))
bars = ax.barh(edu_counts.index[::-1], edu_counts.values[::-1], color=colors_edu)
ax.set_title('Education Background Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Count')
for bar, val in zip(bars, edu_counts.values[::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# === Certification Distribution ===
# FINDING: 26 candidates (14.9%) have NO certification. 
# Most common cert: "University project certificate" (25) — typically held by weaker/junior candidates.
# Professional certs like PMP (6), CFA Level I (4), Scrum Master (7) are rarer but signal strength.
# Some candidates have mismatched certs (e.g., AWS cert for a Financial Analyst role).

cert_counts = df['Certification'].value_counts()

fig, ax = plt.subplots(figsize=(12, 7))
colors_cert = sns.color_palette('Set2', len(cert_counts))
bars = ax.barh(cert_counts.index[::-1], cert_counts.values[::-1], color=colors_cert)
ax.set_title('Certification Distribution (excl. missing)', fontsize=14, fontweight='bold')
ax.set_xlabel('Count')
for bar, val in zip(bars, cert_counts.values[::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# === Certification vs Candidate Level ===
# FINDING: Missing certifications are concentrated in Weak and Unusual candidates.
# Strong candidates almost always have a certification.

df['Has Certification'] = df['Certification'].notna()
cert_level = df.groupby('Candidate Level')['Has Certification'].value_counts(normalize=True).unstack(fill_value=0)
cert_level = cert_level.loc[order] * 100

fig, ax = plt.subplots(figsize=(8, 5))
cert_level.plot(kind='bar', stacked=True, color=['#e74c3c', '#2ecc71'], ax=ax, edgecolor='white', linewidth=1)
ax.set_title('Certification Presence by Candidate Level', fontsize=14, fontweight='bold')
ax.set_ylabel('Percentage (%)')
ax.set_xlabel('')
ax.legend(['No Certification', 'Has Certification'], loc='lower right')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 6. Unusual / Surprise Candidates (Edge Cases)These 25 candidates are designed to **stress-test AI evaluation systems**. They have atypical profiles that don't fit neat categories.

In [ ]:
# === Edge Case Categories ===
# FINDING: 9 distinct edge-case types. Most common:
#   - International experience with unfamiliar titles (4)
#   - Off-topic answers despite strong CV (4) 
#   - Strong theory but limited practical delivery (4)
# These test whether an AI can handle nuance beyond simple keyword matching.

surprise = df[df['Candidate Level'] == 'Unusual/Surprise']
print(f"Total Unusual/Surprise candidates: {len(surprise)}")
print(f"Fit Score range: {surprise['Expected Fit Score'].min()} – {surprise['Expected Fit Score'].max()}")
print(f"Mean Fit Score: {surprise['Expected Fit Score'].mean():.1f}")
print(f"Years Experience range: {surprise['Years Experience'].min()} – {surprise['Years Experience'].max()}")
print()

edge_counts = surprise['Surprise / Edge Case'].value_counts()
print("Edge Case Categories:")
for case, count in edge_counts.items():
    print(f"  [{count}] {case}")

In [ ]:
# === Edge Case Fit Score Distribution ===
# FINDING: Wide spread (45-80), showing these are genuinely ambiguous profiles.
# Some score as high as average-strong candidates, others near weak territory.

fig, ax = plt.subplots(figsize=(10, 5))
edge_data = surprise.groupby('Surprise / Edge Case')['Expected Fit Score'].mean().sort_values(ascending=True)

colors_edge = sns.color_palette('Spectral', len(edge_data))
bars = ax.barh(range(len(edge_data)), edge_data.values, color=colors_edge)
ax.set_yticks(range(len(edge_data)))
ax.set_yticklabels([t[:60] + '...' if len(t) > 60 else t for t in edge_data.index], fontsize=9)
ax.set_title('Mean Fit Score by Edge Case Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Mean Fit Score')
for bar, val in zip(bars, edge_data.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.0f}', va='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Tools & Skills Analysis

In [ ]:
# === Most Mentioned Tools ===
# FINDING: Excel, PowerPoint, and Jira are the most frequently listed tools.
# Tool preferences vary by role (e.g., SQL/Python for Data Analysts, Asana/MS Project for PMs)

from collections import Counter

all_tools = []
for tools_str in df['Tools'].dropna():
    all_tools.extend([t.strip() for t in tools_str.split(',')])

tool_counts = Counter(all_tools).most_common(15)
tool_names, tool_vals = zip(*tool_counts)

fig, ax = plt.subplots(figsize=(12, 6))
colors_tools = sns.color_palette('plasma', len(tool_names))
bars = ax.barh(tool_names[::-1], tool_vals[::-1], color=colors_tools)
ax.set_title('Top 15 Most Mentioned Tools', fontsize=14, fontweight='bold')
ax.set_xlabel('Frequency')
for bar, val in zip(bars, tool_vals[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, str(val), va='center', fontweight='bold', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# === Top Tools by Role ===
# FINDING: Clear role-specific tool preferences:
#   - Project Managers: MS Project, Asana, Jira, PowerPoint
#   - Data Analysts: SQL, Python, Tableau, Power BI
#   - Backend Developers: Git, Docker, PostgreSQL, Django/Flask
#   - Financial Analysts: Excel, Bloomberg, SAP
#   - Marketing Specialists: Google Ads, Hootsuite, Canva

from collections import Counter

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()

for idx, role in enumerate(df['Role'].unique()):
    role_tools = []
    for t in df[df['Role'] == role]['Tools'].dropna():
        role_tools.extend([x.strip() for x in t.split(',')])
    top10 = Counter(role_tools).most_common(8)
    names, vals = zip(*top10)
    colors_r = sns.color_palette('crest', len(names))
    axes[idx].barh(names[::-1], vals[::-1], color=colors_r)
    axes[idx].set_title(role, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Frequency')

axes[5].axis('off')
plt.suptitle('Top Tools by Role', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 8. Summary of Key Findings### 📋 Dataset Structure- **175 synthetic candidate profiles** across **5 roles** and **5 fields** (35 each — perfectly balanced)- **4 candidate levels**: Strong (50), Average (50), Weak (50), Unusual/Surprise (25)- All profiles are synthetic — no real personal data### 📊 Fit Score Insights- Scores range from **25 to 95** (mean = 63.2)- **Clear separation by level**: Strong (82–95), Average (58–76), Weak (25–49), Unusual (45–80)- Fit scores are **consistent across roles** — no single role is scored higher or lower- **Strong positive correlation** between Years Experience and Fit Score (r = 0.69)### 🎓 Education & Certifications- **21 unique education backgrounds** — diverse mix of technical and business degrees- **14.9% of candidates lack certifications** (concentrated in Weak/Unusual levels)- Certification mismatches exist (e.g., AWS cert for non-tech roles) — tests AI evaluation logic### ⚡ Unusual/Surprise Candidates (Edge Cases)- **25 candidates** designed to stress-test AI systems- **9 edge-case categories** including career switchers, overqualified applicants, and theory-vs-practice gaps- Wide fit score spread (45–80) — intentionally ambiguous- These candidates **break the Experience → Fit Score pattern** (some have 12 years but modest scores)### 🛠️ Tools- **Excel, PowerPoint, and Jira** are the most universally mentioned tools- Clear **role-specific tool clusters** (e.g., SQL/Python for data roles, Git/Docker for backend)### 🎯 Design IntentThis dataset is clearly designed as a **benchmarking dataset for AI-driven candidate evaluation**. The balanced structure, synthetic nature, and intentional edge cases suggest it is meant to test:1. Whether an AI can correctly classify candidates by strength level2. How well it handles ambiguous/unusual profiles3. Whether it relies on simple heuristics (years = good) or deeper reasoning